# 🚀 Full Pipeline: arXiv CS Papers (Data Prep + Embedding)

**Notebook này chạy toàn bộ pipeline trên Google Colab:**
1. Download dataset arXiv từ Kaggle (~4.8 GB)
2. Lọc và làm sạch dữ liệu (chỉ lấy CS papers)
3. Tạo embeddings bằng GPU (all-MiniLM-L6-v2)
4. Download kết quả về local hoặc save lên Google Drive

**Yêu cầu:**
- Google Colab với GPU T4 (Runtime → Change runtime type → T4 GPU)
- Kaggle API key (hoặc upload file thủ công)

**Thời gian ước tính (GPU T4):**
- Data Prep: ~5-10 phút
- Embedding 1.16M papers: ~25-35 phút
- Tổng: **~40-45 phút**

## 1. Setup & Install

In [ ]:
!pip install -q sentence-transformers torch kaggle tqdm

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

## 2. Download Dataset từ Kaggle

### Option A: Dùng Kaggle API (khuyến nghị)
1. Vào https://www.kaggle.com/settings → API → Create New Token
2. Upload file `kaggle.json` khi được yêu cầu

### Option B: Upload thủ công
- Tải file từ https://www.kaggle.com/datasets/Cornell-University/arxiv
- Upload lên Colab hoặc Google Drive

In [ ]:
# ====== CONFIG ======
USE_DRIVE = True  # True = save to Google Drive (recommended for large files)

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/BigData'
    os.makedirs(DATA_DIR, exist_ok=True)
else:
    DATA_DIR = '/content'

RAW_FILE = os.path.join(DATA_DIR, 'arxiv-metadata-oai-snapshot.json')
CLEAN_FILE = os.path.join(DATA_DIR, 'arxiv_cs_full.jsonl')
OUTPUT_FILE = os.path.join(DATA_DIR, 'arxiv_cs_full_with_vectors.jsonl')

print(f"Data dir: {DATA_DIR}")
print(f"Raw file: {RAW_FILE}")
print(f"Clean file: {CLEAN_FILE}")
print(f"Output file: {OUTPUT_FILE}")

In [ ]:
# Download từ Kaggle (chỉ cần chạy 1 lần)
if not os.path.exists(RAW_FILE):
    from google.colab import files
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    !kaggle datasets download -d Cornell-University/arxiv -p {DATA_DIR}
    !cd {DATA_DIR} && unzip -q arxiv.zip && rm arxiv.zip
    print(f"\nDownloaded! Size: {os.path.getsize(RAW_FILE) / 1024**3:.1f} GB")
else:
    print(f"File already exists: {os.path.getsize(RAW_FILE) / 1024**3:.1f} GB")

## 3. Data Preparation (Lọc & Làm sạch)

- Lọc chỉ giữ bài báo Computer Science (`cs.*`)
- Parse year từ arXiv ID (chính xác hơn `update_date`)
- Streaming: đọc từng dòng, không load toàn bộ file vào RAM
- Output: JSONL format

In [ ]:
import json
import time
from tqdm.auto import tqdm

def parse_year_from_arxiv_id(arxiv_id):
    """Parse publication year from arXiv ID.
    New format: YYMM.nnnnn (e.g., '2301.00001' -> 2023)
    Old format: cs/0112017 -> fallback to None
    """
    if '.' in arxiv_id:
        prefix = arxiv_id.split('.')[0]
        if '/' in prefix:
            prefix = prefix.split('/')[-1]
        if len(prefix) == 4 and prefix.isdigit():
            yy = int(prefix[:2])
            return str(2000 + yy) if yy < 90 else str(1900 + yy)
    return None


def extract_cs_papers(input_file, output_file):
    """Extract all CS papers from arXiv dataset using streaming."""
    print(f"Input:  {input_file}")
    print(f"Output: {output_file}")
    print("Streaming full dataset, filtering CS papers...\n")

    total_scanned = 0
    cs_count = 0
    skipped = 0
    start = time.time()

    with open(input_file, 'r', encoding='utf-8') as fin, \
         open(output_file, 'w', encoding='utf-8') as fout:
        for line in tqdm(fin, desc='Scanning', unit=' lines'):
            total_scanned += 1
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                skipped += 1
                continue

            categories = record.get('categories', '')
            if 'cs.' not in categories:
                continue

            title = record.get('title', '').replace('\n', ' ').strip()
            abstract = record.get('abstract', '').replace('\n', ' ').strip()

            if not title or not abstract:
                skipped += 1
                continue

            arxiv_id = record.get('id', '')
            year = parse_year_from_arxiv_id(arxiv_id)
            if year is None:
                update_date = record.get('update_date', '')
                year = update_date.split('-')[0] if update_date else None

            clean = {
                'id': arxiv_id,
                'title': title,
                'abstract': abstract,
                'categories': categories,
                'authors': record.get('authors', ''),
                'year': year,
            }

            fout.write(json.dumps(clean, ensure_ascii=False) + '\n')
            cs_count += 1

    elapsed = time.time() - start
    output_size = os.path.getsize(output_file) / (1024 * 1024)

    print(f"\n✅ Done in {elapsed:.1f}s")
    print(f"  Scanned:  {total_scanned:,} records")
    print(f"  CS found: {cs_count:,} papers")
    print(f"  Skipped:  {skipped:,} (bad JSON or missing title/abstract)")
    print(f"  Output:   {output_file} ({output_size:.1f} MB)")
    return cs_count

In [ ]:
if not os.path.exists(CLEAN_FILE):
    cs_count = extract_cs_papers(RAW_FILE, CLEAN_FILE)
else:
    with open(CLEAN_FILE, 'r', encoding='utf-8') as f:
        cs_count = sum(1 for _ in f)
    print(f"Clean file already exists: {cs_count:,} records ({os.path.getsize(CLEAN_FILE) / 1024**2:.1f} MB)")

with open(CLEAN_FILE, 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())
print(f"\nSample record:")
print(f"  ID: {sample['id']}")
print(f"  Year: {sample['year']}")
print(f"  Title: {sample['title'][:80]}...")
print(f"  Categories: {sample['categories']}")

## 4. Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'all-MiniLM-L6-v2'
model = SentenceTransformer(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Device: {model.device}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"Max seq length: {model.max_seq_length} tokens")

## 5. Sanity Check

In [ ]:
from sentence_transformers.util import cos_sim

test_pairs = [
    ('deep learning for image classification', 'using neural networks to categorize pictures'),
    ('reinforcement learning in robotics', 'training robots with reward-based algorithms'),
    ('natural language processing', 'quantum computing algorithms'),
]

print("Cosine Similarity Sanity Check:")
for a, b in test_pairs:
    emb = model.encode([a, b])
    sim = cos_sim(emb[0], emb[1]).item()
    label = '✅ HIGH' if sim > 0.5 else ('⚠️ MED' if sim > 0.3 else '❌ LOW')
    print(f"  {label} {sim:.4f}  |  \"{a}\"  vs  \"{b}\"")

## 6. Run Embedding (Main)

- **GPU T4**: ~512 docs/batch, ~500-800 docs/sec
- Checkpoint mỗi 50,000 records (phòng crash)
- Hỗ trợ resume nếu bị ngắt
- **Truncate abstract** to ~960 chars (~240 tokens) to fit model's 256-token limit

In [ ]:
BATCH_SIZE = 512
CHECKPOINT_EVERY = 50_000
PART_SIZE = 200_000  # Chia thành các file nhỏ 200k tài liệu (~900MB/file)
PROGRESS_FILE = OUTPUT_FILE + '.progress.json'

import glob
import re
import os
import json

def count_lines(path):
    if not os.path.exists(path):
        return 0
    with open(path, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

def count_already_done_parts(output_file_path):
    base_path = output_file_path.replace('.jsonl', '')
    pattern = f"{base_path}_part_*.jsonl"
    files = glob.glob(pattern)
    total_lines = 0
    
    part_files = []
    for f in files:
        match = re.search(r'_part_(\d+)\.jsonl$', f)
        if match:
            part_num = int(match.group(1))
            part_files.append((part_num, f))
    
    part_files.sort()
    
    if not part_files:
        return 0
        
    print("Các part file đã tạo trên Google Drive:")
    for part_num, filepath in part_files:
        lines = count_lines(filepath)
        size_mb = os.path.getsize(filepath) / 1024**2
        print(f"  Part {part_num}: {lines:,} dòng ({size_mb:.1f} MB)")
        total_lines += lines
    return total_lines

# Tóm tắt abstract
MAX_ABSTRACT_CHARS = 960
def truncate_abstract(abstract, max_chars=MAX_ABSTRACT_CHARS):
    if len(abstract) <= max_chars:
        return abstract
    cut = abstract[:max_chars]
    last_period = cut.rfind('. ')
    if last_period > max_chars * 0.6:
        return cut[:last_period + 1]
    return cut

print("Counting records in clean file...")
total = count_lines(CLEAN_FILE)

print("\nChecking existing part files on Google Drive...")
already_done = count_already_done_parts(OUTPUT_FILE)

if already_done > total:
    already_done = total

print(f"\nTotal: {total:,}  |  Done: {already_done:,}  |  Remaining: {total - already_done:,}")

# Show truncation stats on a sample
trunc_count = 0
sample_n = min(1000, total)
with open(CLEAN_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= sample_n:
            break
        rec = json.loads(line)
        if len(rec.get('abstract', '')) > MAX_ABSTRACT_CHARS:
            trunc_count += 1
print(f"Truncation preview: {trunc_count}/{sample_n} abstracts ({trunc_count/sample_n*100:.1f}%) will be truncated at {MAX_ABSTRACT_CHARS} chars")

In [ ]:
# === AUTO KEEP-ALIVE (Browser Injection) ===
try:
    from IPython.display import display, HTML
    js_code = '''
    <script>
    function keepAlive() {
        console.log("Colab Keep-Alive Active - clicking connect button...");
        var button = document.querySelector("colab-connect-button");
        if (button) { button.click(); }
        var shadowBtn = document.querySelector("#connect");
        if (shadowBtn) { shadowBtn.click(); }
    }
    if (window.keepAliveInterval) { clearInterval(window.keepAliveInterval); }
    window.keepAliveInterval = setInterval(keepAlive, 60000);
    console.log("Colab Keep-Alive script injected successfully.");
    </script>
    '''
    display(HTML(js_code))
except Exception as e:
    print("Warning: Could not inject Keep-Alive JS:", e)

# === MAIN EMBEDDING LOOP WITH PART-SPLITTING ===
import time
import math
import os
import json
from tqdm.auto import tqdm

texts_batch = []
records_batch = []
processed_total = already_done
skipped = 0
truncated_count = 0
start_time = time.time()

def get_part_file_path(output_file_path, doc_index, part_size):
    base_path = output_file_path.replace('.jsonl', '')
    part_num = (doc_index // part_size) + 1
    return f"{base_path}_part_{part_num}.jsonl"

def check_ends_with_newline(file_path):
    if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
        return True
    try:
        with open(file_path, 'rb') as f:
            f.seek(-1, os.SEEK_END)
            return f.read(1) == b'\n'
    except Exception:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                if lines:
                    return lines[-1].endswith('\n')
        except Exception:
            pass
    return True

pbar = tqdm(total=total, initial=already_done, desc='Embedding', unit='doc')
current_file_path = None
out_file = None

try:
    with open(CLEAN_FILE, 'r', encoding='utf-8') as fin:
        for line in fin:
            skipped += 1
            if skipped <= already_done:
                continue

            record = json.loads(line.strip())
            title = record.get('title', '')
            abstract = record.get('abstract', '')

            if len(abstract) > MAX_ABSTRACT_CHARS:
                truncated_count += 1
            abstract_trunc = truncate_abstract(abstract)
            text = f"{title} [SEP] {abstract_trunc}"

            texts_batch.append(text)
            records_batch.append(record)

            if len(texts_batch) >= BATCH_SIZE:
                target_file_path = get_part_file_path(OUTPUT_FILE, processed_total, PART_SIZE)
                
                if target_file_path != current_file_path:
                    if out_file is not None:
                        out_file.flush()
                        out_file.close()
                    current_file_path = target_file_path
                    ends_with_nl = check_ends_with_newline(current_file_path)
                    out_file = open(current_file_path, 'a', encoding='utf-8')
                    if not ends_with_nl:
                        print(f"\nFixing missing newline at end of {os.path.basename(current_file_path)}")
                        out_file.write('\n')
                    print(f"\nChuyển sang ghi file: {os.path.basename(current_file_path)}")

                embeddings = model.encode(
                    texts_batch,
                    show_progress_bar=False,
                    normalize_embeddings=True,
                    batch_size=BATCH_SIZE,
                )
                for rec, emb in zip(records_batch, embeddings):
                    rec['embedding'] = emb.tolist()
                    out_file.write(json.dumps(rec, ensure_ascii=False) + '\n')

                prev_processed = processed_total
                processed_total += len(texts_batch)
                pbar.update(len(texts_batch))
                texts_batch.clear()
                records_batch.clear()

                # Robust boundary checkpoint check
                if (processed_total // CHECKPOINT_EVERY) > (prev_processed // CHECKPOINT_EVERY):
                    if out_file is not None:
                        out_file.flush()
                    with open(PROGRESS_FILE, 'w') as pf:
                        json.dump({'already_done': processed_total}, pf)
                    elapsed = time.time() - start_time
                    speed = (processed_total - already_done) / elapsed if elapsed > 0 else 0
                    print(f"  Checkpoint: {processed_total:,} done, {speed:.0f} docs/sec, {truncated_count:,} truncated")

    if texts_batch:
        target_file_path = get_part_file_path(OUTPUT_FILE, processed_total, PART_SIZE)
        if target_file_path != current_file_path:
            if out_file is not None:
                out_file.flush()
                out_file.close()
            current_file_path = target_file_path
            ends_with_nl = check_ends_with_newline(current_file_path)
            out_file = open(current_file_path, 'a', encoding='utf-8')
            if not ends_with_nl:
                out_file.write('\n')
            print(f"\nGhi lô cuối cùng vào: {os.path.basename(current_file_path)}")
        
        embeddings = model.encode(
            texts_batch,
            show_progress_bar=False,
            normalize_embeddings=True,
            batch_size=BATCH_SIZE,
        )
        for rec, emb in zip(records_batch, embeddings):
            rec['embedding'] = emb.tolist()
            out_file.write(json.dumps(rec, ensure_ascii=False) + '\n')
        processed_total += len(texts_batch)
        pbar.update(len(texts_batch))

    pbar.close()

finally:
    if out_file is not None:
        out_file.flush()
        out_file.close()
    with open(PROGRESS_FILE, 'w') as pf:
        json.dump({'already_done': processed_total}, pf)

elapsed = time.time() - start_time
speed = (processed_total - already_done) / elapsed if elapsed > 0 else 0
print(f"\n✅ Hoàn thành! {processed_total:,} tài liệu trong {elapsed:.1f}s ({speed:.0f} docs/sec)")
print(f"Số lượng abstract bị truncate: {truncated_count:,} ({truncated_count/max(processed_total-already_done,1)*100:.1f}%)")


## 7. Verify Output

In [ ]:
verify_count = 0
sample = None
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        verify_count += 1
        if verify_count == 1:
            sample = json.loads(line)

print(f"Total records in output: {verify_count:,}")
print(f"Output file size: {os.path.getsize(OUTPUT_FILE) / (1024**3):.2f} GB")
print(f"\nSample record keys: {list(sample.keys())}")
print(f"Embedding dim: {len(sample['embedding'])}")
print(f"First 5 values: {sample['embedding'][:5]}")
print(f"Year: {sample['year']}")
print(f"Title: {sample['title'][:80]}...")

import numpy as np
norm = np.linalg.norm(sample['embedding'])
print(f"\nVector norm: {norm:.6f} (should be ~1.0)")
print(f"Normalized: {'YES ✅' if abs(norm - 1.0) < 0.01 else 'NO ❌'}")

## 8. Save to Google Drive

Nếu `USE_DRIVE=True`, file đã ghi trực tiếp vào Drive.  
Nếu `USE_DRIVE=False`, chạy cell này để copy:

In [ ]:
if not USE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    dest = '/content/drive/MyDrive/BigData/'
    os.makedirs(dest, exist_ok=True)
    shutil.copy(OUTPUT_FILE, dest)
    shutil.copy(CLEAN_FILE, dest)
    print(f'Copied to Google Drive: {dest}')
else:
    print(f'Files are already on Google Drive at: {DATA_DIR}')
    print(f'  Clean: {CLEAN_FILE}')
    print(f'  Vectors: {OUTPUT_FILE}')

---
## 📝 Sau khi chạy xong

1. **Download file** `arxiv_cs_full_with_vectors.jsonl` từ Google Drive về máy local
2. Đặt vào thư mục `data/` của project
3. Chạy indexing:
```bash
# Tăng RAM cho ES (cần 4GB cho full dataset)
ES_HEAP=4g docker compose up -d

# Index 100k (để test nhanh)
head -n 100000 data/arxiv_cs_full_with_vectors.jsonl > data/arxiv_cs_100k_with_vectors.jsonl
python src/pipeline/index_data_v2.py --input data/arxiv_cs_100k_with_vectors.jsonl --index arxiv_papers_v2_100k

# Index full (nếu đủ RAM)
python src/pipeline/index_data_v2.py --input data/arxiv_cs_full_with_vectors.jsonl --index arxiv_papers_v2_full
```

## 9. Ghép các Part file thành file duy nhất (Merge Parts)

Sau khi chạy xong tất cả các part, bạn có thể ghép các file nhỏ đó lại thành một file duy nhất trên Google Drive bằng câu lệnh Linux dưới đây:

In [ ]:
# Ghép các part (sẽ ghép theo thứ tự tên file part_1, part_2,...)
!cat {DATA_DIR}/arxiv_cs_full_with_vectors_part_*.jsonl > {OUTPUT_FILE}

# Kiểm tra số dòng để đảm bảo đủ 1.2M dòng
print("Đang đếm số dòng trong file đã ghép...")
!wc -l {OUTPUT_FILE}

# Kiểm tra dung lượng file cuối cùng
!ls -lh {OUTPUT_FILE}